### Initialization

In [0]:
import pyspark.sql.functions as F


### Reading from bronze

In [0]:
df = spark.table("workspace.bronze.crm_sales_details")
print(f"total number of rows:{df.count()}")
df.display()

### Silver Transformations

### Trimming

In [0]:
from pyspark.sql.types import StringType,DataType,DateType

for field in df.schema.fields:
    if isinstance(field.dataType,StringType):
        df = df.withColumn(field.name,F.trim(F.col(field.name)))


### cleaning

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, length

df = (
    df
    .withColumn(
        "sls_order_dt",
        F.when(
            (col("sls_order_dt") == 0) | (length(col("sls_order_dt")) != 8),
            None
        ).otherwise(
            F.to_date(col("sls_order_dt").cast("string"), "yyyyMMdd")
        )
    )
    .withColumn(
        "sls_ship_dt",
        F.when(
            (col("sls_ship_dt") == 0) | (length(col("sls_ship_dt")) != 8),
            None
        ).otherwise(
            F.to_date(col("sls_ship_dt").cast("string"), "yyyyMMdd")
        )
    )
    .withColumn(
        "sls_due_dt",
        F.when(
            (col("sls_due_dt") == 0) | (length(col("sls_due_dt")) != 8),
            None
        ).otherwise(
            F.to_date(col("sls_due_dt").cast("string"), "yyyyMMdd")
        )
    )
)



### Sales and Price Corrections

In [0]:
df = (
    df.withColumn(
        "sls_price",
        F.when(
            (col("sls_price").isNull()) | (col("sls_price") <= 0),
            F.when(
                col("sls_quantity") != 0,
                col("sls_sales") / col("sls_quantity")
            ).otherwise(None)

        ).otherwise(col("sls_price"))
    )
)
print("Rows with still-null price after fix: ",
      df.filter(F.col("sls_price").isNull()).count())

In [0]:
rename_map ={
     "sls_ord_num": "order_number",
    "sls_prd_key": "product_number",
    "sls_cust_id": "customer_id",
    "sls_order_dt": "order_date",
    "sls_ship_dt": "ship_date",
    "sls_due_dt": "due_date",
    "sls_sales": "sales_amount",
    "sls_quantity": "quantity",
    "sls_price": "price"
}
for old_name,new_name in rename_map.items():
    df = df.withColumnRenamed(old_name,new_name)

### ## Sanity checks of dataframe

In [0]:
print("Sample data :")
df.limit(10).display()

print("\n Date range of orders:")
df.select(
    F.min("order_date").alias("earliest_order"),
    F.max("order_date").alias("latest_order")
).display()

print("\n Null count check:")
df.select([F.count(F.when(F.col(c).isNull(),c)).alias(c)for c in df.columns]).display()

Writing Silver Table

In [0]:
%sql
drop table workspace.silver.crm_sales

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.crm_sales")

In [0]:
%sql
select * from workspace.silver.crm_sales